# Lab 1 — Images are arrays: NumPy & OpenCV basics

**Deep Learning for Computer Vision — Day 1** · ECAM, 5MAU · 2026–2027

**Goals.** After this lab you can

1. load, display and inspect an image (shape, `dtype`, channels, BGR vs RGB);
2. access pixels and regions with NumPy slicing, using the right (row, column) convention;
3. avoid the classic `uint8` overflow pitfalls;
4. compute and interpret histograms and equalise them;
5. segment an object by colour in the HSV space — the first step of our *pixel → robot* pipeline.

**Duration:** ~60 min. Cells marked `# TODO` are for you. Cells starting with `assert` check your answer.

**Data.** `table/*.png` are synthetic images of the work table rendered with a simulated wide-angle USB camera (`tools/make_data.py`).

> *References:* Q. Lurkin, *Image Processing – Lab 2: Filtering*, ECAM, https://quentin.lurkin.xyz/courses/imageprocessing/lab2/ · R. Szeliski, *Computer Vision: Algorithms and Applications*, 2nd ed., 2022, ch. 2–3 · OpenCV documentation, https://docs.opencv.org/

In [ ]:
# Install the packages of requirements.txt into the Python environment of this kernel
# (a few minutes the first time, a few seconds afterwards). If something was installed,
# restart the kernel (Kernel > Restart) and reload the page before going on, so that the sliders work.
from pathlib import Path

REQ = next(p for p in [Path("requirements.txt"), Path("../requirements.txt"), Path("../../requirements.txt")] if p.exists())
%pip install -q --disable-pip-version-check -r {REQ.as_posix()}

# Setup check: are all the packages of requirements.txt installed in this kernel?
import importlib
import importlib.util

importlib.invalidate_caches()
REQUIRED = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "cv2": "opencv-contrib-python",
            "ipywidgets": "ipywidgets", "ipympl": "ipympl", "serial": "pyserial"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if "opencv-contrib-python" not in missing:
    import cv2
    if not hasattr(cv2, "aruco"):  # opencv-python (without contrib) shadows opencv-contrib-python
        missing.append("opencv-contrib-python (run `%pip uninstall -y opencv-python opencv-contrib-python` "
                       "then `%pip install \"opencv-contrib-python<5\"`)")
assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                     "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
print("Setup OK: all required packages are installed.")

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

DATA = Path("data") if Path("data").exists() else Path("../data")
plt.rcParams["figure.figsize"] = (10, 5)
print("OpenCV", cv2.__version__, "| NumPy", np.__version__)

## 1. Loading and displaying an image

`cv2.imread` returns a NumPy array of shape `(height, width, channels)` with `dtype=uint8` (values 0–255).
**Watch out:** OpenCV stores the channels in **BGR** order, whereas Matplotlib expects **RGB**.

In [ ]:
img = cv2.imread(str(DATA / "table" / "table_normal.png"))
print("type :", type(img))
print("shape:", img.shape, " -> (rows = height, columns = width, channels)")
print("dtype:", img.dtype, " min:", img.min(), " max:", img.max())

plt.imshow(img)  # wrong colours on purpose!
plt.title("Displayed as if it were RGB")
plt.axis("off");

### Exercise 1.1 — Fix the colours
Convert `img` to RGB in **two** ways: with `cv2.cvtColor` and with pure NumPy slicing (reverse the last axis).

In [ ]:
# TODO
img_rgb = ...   and   img_rgb2 = ...

assert np.array_equal(img_rgb, img_rgb2), "both conversions should give the same array"
plt.imshow(img_rgb)
plt.axis("off");

## 2. Pixels, rows and columns

An image is indexed `img[row, column]` = `img[y, x]`. The origin is the **top-left** corner, `y` grows **downwards**.
This is the opposite order of the `(x, y)` convention used by most OpenCV drawing functions (`cv2.circle(img, (x, y), ...)`) — a very common source of bugs.

In [ ]:
x, y = 500, 240  # a pixel on the red disc
print("BGR value at (x=500, y=240):", img[y, x])

### Exercise 2.1 — Region of interest (ROI)
1. Extract the region `x ∈ [430, 570)`, `y ∈ [180, 300)` into `roi` (a **view**, no copy).
2. Paint a 20×20 white square in the top-left corner of a **copy** of the image, using slicing only.
3. Why does modifying `roi` in place also modify `img`? Check with `np.shares_memory`.

In [ ]:
# TODO
roi = ...  ;  painted = ...

assert roi.shape == (120, 140, 3)
assert (painted[:20, :20] == 255).all() and not (img[:20, :20] == 255).all()
print("roi shares memory with img:", np.shares_memory(roi, img))
fig, ax = plt.subplots(1, 2)
ax[0].imshow(roi[:, :, ::-1]); ax[0].set_title("ROI")
ax[1].imshow(painted[:, :, ::-1]); ax[1].set_title("painted copy");

## 3. Grayscale conversion

The luminance is a weighted sum of the channels (ITU-R BT.601): $Y = 0.299\,R + 0.587\,G + 0.114\,B$. Green has the largest weight because the eye is most sensitive to it.

### Exercise 3.1
Implement `to_gray(img_bgr)` with NumPy (careful with the channel order and the `dtype`) and compare to `cv2.cvtColor(..., cv2.COLOR_BGR2GRAY)`.

In [ ]:
def to_gray(img_bgr):
    # TODO
    ...


gray = to_gray(img)
gray_cv = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
print("max abs difference with OpenCV:", np.abs(gray.astype(int) - gray_cv.astype(int)).max())
assert gray.dtype == np.uint8 and gray.shape == img.shape[:2]
assert np.abs(gray.astype(int) - gray_cv.astype(int)).max() <= 1
plt.imshow(gray, cmap="gray"); plt.axis("off");

## 4. The `uint8` trap

`uint8` arithmetic is **modulo 256**: there is no saturation.

In [ ]:
a = np.array([[200]], dtype=np.uint8)  # a 1x1 image
b = np.array([[100]], dtype=np.uint8)
print("200 + 100 =", (a + b).item(), "  (NumPy, wraps around)")
print("cv2.add   =", cv2.add(a, b).item(), " (OpenCV, saturates)")
print("100 - 200 =", (b - a).item())

### Exercise 4.1 — Brighten the dark image
`table_dark.png` was taken with a too short exposure. Brighten it by adding 200 to every pixel:
1. naively with `+` (observe the artefacts),
2. correctly, by computing in a larger type and clipping to [0, 255].

In [ ]:
dark = cv2.imread(str(DATA / "table" / "table_dark.png"))
naive = dark + np.uint8(200)
# TODO
correct = ...

assert correct.dtype == np.uint8 and correct.min() >= 200
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a_, im, t in zip(ax, [dark, naive, correct], ["dark", "naive + 200", "clipped + 200"]):
    a_.imshow(im[:, :, ::-1]); a_.set_title(t); a_.axis("off")

Adding a constant only shifts the histogram; the **contrast** is unchanged. Section 5 shows better tools.

## 5. Histograms and equalisation

The histogram counts how many pixels take each grey level. It tells you at a glance whether the image is under/over-exposed or has low contrast — **check it when you tune your camera exposure for the project**.

### Exercise 5.1 — Compute a histogram
Write `histogram(gray)` returning an array `h` of 256 counts, **without loops** (hint: `np.bincount` with `minlength`).

In [ ]:
def histogram(gray):
    # TODO
    ...


images = {name: cv2.imread(str(DATA / "table" / f"table_{name}.png"), cv2.IMREAD_GRAYSCALE)
          for name in ["dark", "normal", "bright"]}
h = histogram(images["normal"])
assert h.shape == (256,) and h.sum() == images["normal"].size
assert np.array_equal(h, cv2.calcHist([images["normal"]], [0], None, [256], [0, 256]).ravel())

fig, ax = plt.subplots(2, 3, figsize=(15, 6))
for k, (name, g) in enumerate(images.items()):
    ax[0, k].imshow(g, cmap="gray", vmin=0, vmax=255); ax[0, k].set_title(name); ax[0, k].axis("off")
    ax[1, k].bar(np.arange(256), histogram(g), width=1); ax[1, k].set_xlim(0, 255)
plt.tight_layout()

### Exercise 5.2 — Histogram equalisation
Equalisation maps each grey level through the normalised **cumulative histogram** (CDF), which spreads the levels over [0, 255]:

$$ g' = \mathrm{round}\Big(255 \cdot \frac{\mathrm{CDF}(g) - \mathrm{CDF}_{\min}}{N - \mathrm{CDF}_{\min}}\Big) $$

where $N$ is the number of pixels and $\mathrm{CDF}_{\min}$ the first non-zero value of the CDF. Implement it with a **look-up table** (LUT) and compare to `cv2.equalizeHist`.

In [ ]:
def equalize(gray):
    # TODO
    ...


eq = equalize(images["dark"])
eq_cv = cv2.equalizeHist(images["dark"])
print("max difference with OpenCV:", np.abs(eq.astype(int) - eq_cv.astype(int)).max())
assert np.abs(eq.astype(int) - eq_cv.astype(int)).max() <= 1
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].imshow(images["dark"], cmap="gray", vmin=0, vmax=255); ax[0].set_title("dark")
ax[1].imshow(eq, cmap="gray"); ax[1].set_title("equalised")
ax[2].plot(np.cumsum(histogram(images["dark"])), label="CDF dark")
ax[2].plot(np.cumsum(histogram(eq)), label="CDF equalised"); ax[2].legend();

**Question.** Software corrections cannot recover information that was lost by the sensor. Which information is lost in `table_bright.png` and cannot be recovered? What should you do *on the camera* instead?

## 6. Colour segmentation: RGB vs HSV

To locate the red parts on the table we want a **binary mask** that is `True` on red pixels.

### Exercise 6.1 — Thresholding in RGB
Build a mask of red pixels on `table_normal.png` with a rule on the B, G, R values, then apply the **same rule** to `table_dark.png`. What happens?

In [ ]:
def red_mask_bgr(im):
    b, g, r = [im[:, :, k].astype(int) for k in range(3)]
    # TODO
    return ...


normal = cv2.imread(str(DATA / "table" / "table_normal.png"))
fig, ax = plt.subplots(1, 2)
ax[0].imshow(red_mask_bgr(normal), cmap="gray"); ax[0].set_title("normal exposure")
ax[1].imshow(red_mask_bgr(dark), cmap="gray"); ax[1].set_title("dark exposure");

In **HSV**, the colour itself (**H**ue) is separated from its purity (**S**aturation) and brightness (**V**alue). A change of exposure mostly affects V, so a rule on H and S is much more robust.

In OpenCV (8-bit images): `H ∈ [0, 179]` (degrees / 2), `S, V ∈ [0, 255]`. **Red sits at both ends of the hue circle** (around 0 *and* 179), so it needs two ranges.

In [ ]:
hsv = cv2.cvtColor(normal, cv2.COLOR_BGR2HSV)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a_, k, name in zip(ax, range(3), "HSV"):
    a_.imshow(hsv[:, :, k], cmap="hsv" if name == "H" else "gray"); a_.set_title(name); a_.axis("off")

### Exercise 6.2 — Thresholding in HSV
Write `red_mask_hsv(im_bgr)` with `cv2.inRange` (two hue ranges combined with `|` or `cv2.bitwise_or`). The minimum value of V should be low so that the dark image still works. Compare the masks on the three exposures.

In [ ]:
def red_mask_hsv(im_bgr):
    hsv = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2HSV)
    # TODO
    ...


bright = cv2.imread(str(DATA / "table" / "table_bright.png"))
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a_, im, t in zip(ax, [dark, normal, bright], ["dark", "normal", "bright"]):
    a_.imshow(red_mask_hsv(im), cmap="gray"); a_.set_title(f"HSV mask – {t}"); a_.axis("off")
m_dark, m_norm = red_mask_hsv(dark) > 0, red_mask_hsv(normal) > 0
iou = (m_dark & m_norm).sum() / (m_dark | m_norm).sum()
print(f"overlap (IoU) between the dark and normal masks: {iou:.2f}")
assert iou > 0.8, "your HSV mask should be robust to the exposure change"

### Exercise 6.3 — Interactive tuning
Use the sliders to find the HSV ranges of the **blue** parts, then of the **yellow** square. Write down the values: you will reuse this tool with the real camera in the project.

In [ ]:
from ipywidgets import IntRangeSlider, interact


@interact(h=IntRangeSlider(min=0, max=179, value=(100, 130)),
          s=IntRangeSlider(min=0, max=255, value=(120, 255)),
          v=IntRangeSlider(min=0, max=255, value=(40, 255)))
def tune(h, s, v):
    mask = cv2.inRange(hsv, (h[0], s[0], v[0]), (h[1], s[1], v[1]))
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    ax[0].imshow(normal[:, :, ::-1]); ax[0].axis("off")
    ax[1].imshow(mask, cmap="gray"); ax[1].axis("off")
    plt.show()

## 7. (Optional) Grab frames from a camera

If you have a USB camera (or your laptop webcam), set `USE_CAMERA = True`. Try your HSV thresholds on a live image and observe the effect of the exposure settings. Not every driver supports every property: check the returned value.

In [ ]:
USE_CAMERA = False
CAMERA_INDEX = 0  # 0 = first camera; try 1 for the USB camera if the laptop has a webcam

if USE_CAMERA:
    cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)  # CAP_DSHOW is faster on Windows
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)
    print("auto exposure:", cap.get(cv2.CAP_PROP_AUTO_EXPOSURE), "exposure:", cap.get(cv2.CAP_PROP_EXPOSURE))
    ok, frame = cap.read()
    cap.release()
    if ok:
        fig, ax = plt.subplots(1, 2, figsize=(14, 4))
        ax[0].imshow(frame[:, :, ::-1]); ax[1].imshow(red_mask_hsv(frame), cmap="gray")